In [ ]:
#!pip install pymupdf --q

In [1]:
import os
import langchain, langchain_community
print(langchain.__version__)
print(langchain_community.__version__)

0.2.17
0.2.19


In [2]:
from langchain_community.document_loaders import PyMuPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import utils

In [3]:
DATA_FOLDER = "documents"
VECTOR_STORE = "hr_vector_db"

In [4]:
loader = DirectoryLoader(
    path= DATA_FOLDER,                 # Path to your local directory
    glob="./*.pdf",                    # Pattern to match PDF files
    loader_cls=PyMuPDFLoader,          # Use PyMuPDF as the underlying parser
    use_multithreading=True,           # Speeds up loading by processing multiple files concurrently
    show_progress=True                 # Displays a loading bar in your terminal
)

In [5]:
documents = loader.load()

100%|██████████| 10/10 [00:00<00:00, 15.67it/s]


In [6]:
text_splitter  = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=50,
    separators=["\n\n", "\n", "(?<=\. )", " ", ""]
)
splitted_text=text_splitter.split_documents(documents)

In [7]:
len(splitted_text)

132

In [8]:
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

In [9]:
vectordb = FAISS.from_documents(
    documents=splitted_text,
    embedding=embeddings
)

In [10]:
retriever = vectordb.as_retriever(search_kwargs={"k": 4})

In [11]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

In [12]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an HR Support Assistant.\n\n"
        "Answer employee questions using ONLY the provided company documents.\n"
        "If the answer is not found in the documents, say:\n"
        '"I’m not sure based on current HR policies."\n\n'
        "Be clear, professional, and policy-aligned.\n\n"
        "HR Context:\n"
        "{context}"
    ),
    (
        "user",
        "{question}"
    )
])

In [13]:
def chat():
    print("\nHR Support Chatbot (type 'exit' to quit)\n")

    while True:
        question = input("Employee: ")
        if question.lower() == "exit":
            break

        docs = retriever.invoke(question)
        context = "\n\n".join(doc.page_content for doc in docs)

        response = llm.invoke(
            prompt.format_messages(
                context=context,
                question=question
            )
        )

        print("\nHR Bot:", response.content)
        print("-" * 60)


In [14]:
chat()


HR Support Chatbot (type 'exit' to quit)


HR Bot: Our leave policies are designed to provide you time to rest, recover, and take care of life outside work. Here are the key points regarding leaves:

1. **Types of Leave**: The company provides various types of leave, including vacation, holidays, sick leave, and other types of time off.

2. **Paid Time Off (PTO) / Vacation**: All eligible employees are entitled to paid vacation time.

3. **Requesting Time Off**: 
   - For planned leave (such as vacation or personal days), use our HR system to submit a leave request. Your manager will approve it online.
   - For extended leaves like parental or medical leave, you’ll work with HR to fill out any additional forms, especially if FMLA applies.
   - For unplanned leave (sick days or emergencies), notify your manager as soon as possible via direct message, phone, or email. The manager or HR can log the leave on your behalf if you are unable to do so.

4. **Unpaid Leave**: 
   - Approval of u